# 🚀 Notebook 03 — XGBoost Health Risk Model Training

Trains the XGBoost health risk prediction model with SHAP explainability.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
print('Ready.')

In [ ]:
from backend.utils.data_generator import generate_health_records

# Generate training data
df = generate_health_records(2000)
print(f'Health records: {df.shape}')
print(f'Risk distribution:\n{df["overall_risk"].value_counts()}')
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

FEATURES = ['pm25', 'pm10', 'co', 'so2', 'no2', 'o3', 'temperature',
            'humidity', 'pressure', 'wind_speed', 'current_aqi',
            'age', 'has_asthma', 'has_heart_condition', 'is_smoker',
            'outdoor_hours', 'exercise_level']

available = [f for f in FEATURES if f in df.columns]
X = df[available].fillna(0)

# Encode risk labels
le = LabelEncoder()
y = le.fit_transform(df['overall_risk'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Classes: {le.classes_}')

In [ ]:
try:
    import xgboost as xgb
    from sklearn.metrics import classification_report, confusion_matrix

    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=42,
    )

    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)

    y_pred = model.predict(X_test)
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    XGB_AVAILABLE = True

except ImportError:
    print('XGBoost not installed. Simulating results...')
    XGB_AVAILABLE = False

In [ ]:
if XGB_AVAILABLE:
    try:
        import shap
        explainer = shap.TreeExplainer(model)
        shap_vals = explainer.shap_values(X_test[:200])
        
        plt.figure(figsize=(12, 6))
        shap.summary_plot(shap_vals, X_test[:200], feature_names=available, show=False)
        plt.tight_layout()
        plt.savefig('../datasets/shap_summary.png', dpi=150, facecolor='#1a1f3a')
        plt.show()
        print('SHAP plots saved!')
    except ImportError:
        print('SHAP not installed — skipping explainability plots')

    # Save model
    os.makedirs('../models', exist_ok=True)
    import pickle
    with open('../models/xgboost_health.pkl', 'wb') as f:
        pickle.dump(model, f)
    with open('../models/health_label_encoder.pkl', 'wb') as f:
        pickle.dump(le, f)
    print('XGBoost model saved to ../models/')
else:
    print('Model not available — using simulated results')

print('\nTraining pipeline complete!')